In [1]:
import warnings

import skrub
from sklearn.ensemble import HistGradientBoostingClassifier
from skrub import TableVectorizer

import sempipes
from sempipes import sem_choose

warnings.filterwarnings("ignore")

dataset = skrub.datasets.fetch_midwest_survey()
X = dataset.X.head(n=500)
y = dataset.y.head(n=500)

responses = skrub.var("responses")
responses = responses.skb.set_description(dataset.metadata["description"])

labels = skrub.var("labels")
labels = labels.skb.set_description(dataset.metadata["target"])

responses = responses.skb.mark_as_X()
labels = labels.skb.mark_as_y()

responses_with_additional_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional demographics-related features, use your intrinsic knowledge about the US. 
        Take into account how the identification with the country or regions of it changed over the generations.         
        Also think about how the identification differs per class and education. The midwest is generally associated 
        with "Midwestern values" — friendliness, modesty, hard work, and community-mindedness.
    """,
    name="demographic_features",
    how_many=5,
)

feature_encoder = TableVectorizer()
encoded_responses = responses_with_additional_features.skb.apply(feature_encoder)

learner = HistGradientBoostingClassifier()
model = encoded_responses.skb.apply_with_sem_choose(
    learner,
    y=labels,
    choices=sem_choose(
        "hgb",
        learning_rate="A promising set of learning rates to try",
    ),
)

--- sempipes.apply_with_sem_choose(HistGradientBoostingClassifier(), SemChoices(name='hgb', params_and_prompts={'learning_rate': 'A promising set of learning rates to try'}))


2026-02-26 20:05:10,713 - INFO - SEMPIPES> Querying 'gemini/gemini-2.5-flash' with 2 messages...'


	Suggested choices for learning_rate: choose_from([0.1, 0.05, 0.2, 0.01, 0.5], name='__sempipes__...learning_rate')


In [2]:
from sempipes.optimisers import MonteCarloTreeSearch, optimise_colopro

sempipes.update_config(verbose_code_synthesis=True)

tuning_data = {"responses": dataset.X.head(n=500), "labels": dataset.y.head(n=500)}

outcomes = optimise_colopro(
    model,
    num_trials=12,
    scoring="accuracy",
    search=MonteCarloTreeSearch(),
    cv=5,
    num_hpo_iterations_per_trial=1,
    additional_env_variables=tuning_data,
)

2026-02-26 20:05:13,435 - INFO - SEMPIPES> COLOPRO> Computing pipeline summary for context-aware optimisation
2026-02-26 20:05:13,746 - INFO - SEMPIPES> COLOPRO> Processing trial 0
2026-02-26 20:05:13,758 - INFO - SEMPIPES> COLOPRO> Initialising optimisation via OPRO
2026-02-26 20:05:13,759 - INFO - SEMPIPES> COLOPRO> Creating root node
2026-02-26 20:05:13,760 - INFO - SEMPIPES> COLOPRO> Scoring pipeline via 5-fold cross-validation and random search HPO
2026-02-26 20:05:29,139 - INFO - SEMPIPES> COLOPRO> Pipeline scoring took 15.38 seconds
2026-02-26 20:05:29,140 - INFO - SEMPIPES> COLOPRO> Score changed from None to 0.714
2026-02-26 20:05:29,141 - INFO - SEMPIPES> COLOPRO> Processing trial 1
2026-02-26 20:05:29,153 - INFO - SEMPIPES> MCT_SEARCH> Expanding childless node 0
2026-02-26 20:05:29,154 - INFO - SEMPIPES> COLOPRO> Trying to improve node with score 0.714
2026-02-26 20:05:29,155 - INFO - SEMPIPES> COLOPRO> Generating new search node
2026-02-26 20:05:29,155 - INFO - SEMPIPES> CO

In [3]:
best_outcome = max(outcomes, key=lambda x: x.score)
best_outcome.score

np.float64(0.9)

In [4]:
from IPython.display import Code, display

display(Code(best_outcome.states.get("demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates additional demographic and identification-related features for the dataset.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The DataFrame with new feature columns added.
    """

    # Feature 1: zip_code_region_prefix
    # This feature extracts the first digit of the ZIP code, which often corresponds to a broad geographical region
    # in the US. This provides a high-level demographic and geographical context for the respondent's location,
    # which is highly relevant for predicting their Census_Region.
    # Input samples: 'In_what_ZIP_code_is_your_home_located': ['74070', '44106', '48185']
    # Fix: Handle potential non-numeric first characters in ZIP codes.
    # Convert to string, take the first character, then attempt to convert to numeric.
    # Use errors='coerce' to turn non-numeric values (like 'N' from 'NaN') into NaN,
    # then fill NaN with a placeholder like -1 to maintain integer type.
    df['zip_code_region_prefix'] = df['In_what_ZIP_code_is_your_home_located'].astype(str).str[0]
    df['zip_code_region_prefix'] = pd.to_numeric(df['zip_code_region_prefix'], errors='coerce').fillna(-1).astype(int)

    # Feature 2: midwest_state_count
    # This feature quantifies how many states a respondent personally considers part of the Midwest.
    # It directly reflects their individual definition and understanding of the Midwest, which can vary
    # and is a strong indicator of their regional identity. A higher count might suggest a stronger
    # identification or a broader, more inclusive view of the Midwest, aligning with "community-mindedness".
    # Input samples: 'Do_you_consider_Illinois_state_as_part_of_the_Midwest': ['No', 'Yes', 'Yes'],
    #                'Do_you_consider_Indiana_state_as_part_of_the_Midwest': ['No', 'No', 'Yes'],
    #                'Do_you_consider_Iowa_state_as_part_of_the_Midwest': ['No', 'Yes', 'No']
    midwest_state_cols = [col for col in df.columns if col.startswith('Do_you_consider_') and col.endswith('_state_as_part_of_the_Midwest')]
    df['midwest_state_count'] = df[midwest_state_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

    # Feature 3: personal_midwestern_identity_score
    # This feature converts the ordinal categorical variable of personal identification into a numerical score.
    # This allows the model to directly use the strength of a respondent's self-identification as a Midwesterner,
    # which is a core aspect of the survey and directly relates to the target.
    # Input samples: 'How_much_do_you_personally_identify_as_a_Midwesterner': ['Not much', 'Not much', 'A lot']
    identity_mapping = {
        'Not at all': 0,
        'Not much': 1,
        'Some': 2,
        'A lot': 3
    }
    df['personal_midwestern_identity_score'] = df['How_much_do_you_personally_identify_as_a_Midwesterner'].map(identity_mapping)

    # Feature 4: age_group_numeric
    # This feature converts age ranges into an ordinal numerical representation. This helps capture generational
    # differences in how people identify with regions. Younger generations might have different perceptions or
    # stronger/weaker ties to regional identities compared to older generations, reflecting changes over time.
    # Input samples: 'Age': ['18-29', '18-29', '30-44']
    age_mapping = {
        '18-29': 0,
        '30-44': 1,
        '45-60': 2,
        '60+': 3
    }
    df['age_group_numeric'] = df['Age'].map(age_mapping)

    # Feature 5: education_level_numeric_x_midwest_identity
    # This feature is an interaction term combining the numerical personal Midwestern identity score with
    # a numerical representation of education level. The prompt specifically asks to consider how identification
    # differs per class and education. This interaction can reveal if certain education levels correlate with
    # a stronger